# Proyecto 1: Clasificación de Phishing en URLs

Este notebook contiene el proceso completo de desarrollo de un modelo de Machine Learning para clasificar URLs como benignas o phishing, cumpliendo con los requisitos del proyecto.

## 1. Contexto y Pregunta de Negocio
**¿Qué predice el modelo?** Predice si una URL ingresada es legítima (benigna) o maliciosa (phishing).

**¿Por qué importa?** El phishing es uno de los ataques cibernéticos más comunes. Bloquear o advertir a los usuarios sobre URLs maliciosas antes de que accedan puede prevenir el robo de credenciales y pérdidas financieras.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import joblib

# Configuraciones gráficas para que los reportes sean fáciles de leer y profesionales
plt.style.use('ggplot')
sns.set_palette('muted')

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
# 2.1 Carga y estructura inicial
# Cargamos los datos del archivo CSV al dataframe de Pandas
df = pd.read_csv('Dataset.csv')
display(df.head())
print("Estructura del dataset:", df.shape)
print("\nInformación general:")
df.info()

In [ ]:
# 2.2 Análisis y manejo de valores nulos
print("Valores nulos por columna:")
display(df.isnull().sum()[df.isnull().sum() > 0])

# Imputar nulos en la columna 'tld' (Top Level Domain) para direcciones IP puras.
# Justificación: Una URL que es una IP (ej: http://192.168.1.1) no tiene dominio de nivel superior (.com, .net).
# Llenamos esos vacíos con 'none' para que el modelo pueda procesarlos sin arrojar error.
if 'tld' in df.columns:
    df['tld'] = df['tld'].fillna('none')
    print("\nNulos imputados con 'none' en la columna 'tld'.")

In [ ]:
# 2.3 Distribución de la variable objetivo (Balance de clases)
# Justificación: Queremos saber cuántas URL benignas vs phishing existen.
# Es fundamental conocer este desbalanceo para decidir qué métricas de evaluación usaremos.
class_counts = df['label'].value_counts()
class_percentages = df['label'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Cantidad': class_counts, 'Porcentaje (%)': class_percentages})
display(dist_df)

In [ ]:
# 2.4 Visualizaciones clave
# Gráfico 1: Distribución de Clases. Gráfico de barras simple y efectivo.
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='label')
plt.title('Distribución de Clases (0: Benigna, 1: Phishing)')
plt.show()

In [ ]:
# Gráfico 2: Heatmap de Correlación
# Justificación: Nos ayuda a ver qué variables numéricas tienen una relación fuerte 
# (positiva o negativa) con que una URL sea Phishing (label).
columnas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
# Tomamos las primeras 10 numéricas y la etiqueta para no saturar visualmente el mapa.
columnas_relevantes = [c for c in columnas_numericas[:10] if c != 'label'] + ['label']

plt.figure(figsize=(10,8))
sns.heatmap(df[columnas_relevantes].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Heatmap de Correlación')
plt.show()

## 3. Extracción, Preprocesamiento y División de Datos

In [ ]:
%%writefile extractor.py
import urllib.parse
import re
import pandas as pd

def extract_features_from_url(url: str) -> pd.DataFrame:
    """
    Dada una URL cruda ingresada por el usuario en nuestra web,
    esta función extrae sus características numéricas (métricas léxicas).
    Es vital para que la App web le dé al modelo los datos en el mismo 
    formato en el que fue entrenado.
    """
    parsed = urllib.parse.urlparse(url)
    
    url_length = len(url)
    domain_length = len(parsed.netloc)
    is_ip = 1 if re.match(r'^[0-9]+(?:\.[0-9]+){3}$', parsed.netloc) else 0
    is_https = 1 if parsed.scheme == 'https' else 0
    
    digits = sum(c.isdigit() for c in url)
    digit_ratio = digits / url_length if url_length > 0 else 0
    
    special_chars = sum(not c.isalnum() for c in url)
    special_char_count = special_chars
    
    # Estas columnas DEBEN ser idénticas a las que entraron al fit() del Pipeline
    features = {
        'url_length': [url_length],
        'domain_length': [domain_length],
        'is_ip': [is_ip],
        'is_https': [is_https],
        'digit_ratio': [digit_ratio],
        'special_char_count': [special_char_count]
    }
    return pd.DataFrame(features)


In [ ]:
# Ahora importamos la función del archivo que acabamos de generar automáticamente
from extractor import extract_features_from_url

url_test = "http://www.ejemplo.com"
features_test = extract_features_from_url(url_test)
print("Variables extraídas para la app web:", list(features_test.columns))
print("Asegúrate de alinear estas columnas con el entrenamiento del modelo.")

In [ ]:
# 3.1 División estratificada de los datos
X = df.drop('label', axis=1)
y = df['label']

# Al poner stratify=y, forzamos a que tanto en el entrenamiento como en las pruebas 
# se mantenga el 14% original de Phishing. Evita sesgos en el entrenamiento.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Tamaño de Entrenamiento:", X_train.shape)
print("Tamaño de Prueba:", X_test.shape)

In [ ]:
# 3.2 Preprocesamiento con ColumnTransformer
# Separamos cuáles variables son numéricas y cuáles son categóricas (texto)
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# El StandardScaler normaliza las variables numéricas para que la Regresión Logística converja más rápido.
# El OneHotEncoder transforma las categorías de texto en columnas de ceros y unos.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

## 4. Modelado y Evaluación: Regresión Logística
En esta fase vamos a entrenar nuestro clasificador principal: **Regresión Logística**. 

Usaremos un `Pipeline` de scikit-learn. Un Pipeline es fundamental en producción porque asegura que las transformaciones aplicadas a los datos de entrenamiento (como el escalado o el OneHotEncoder) se apliquen exactamente igual a los datos de prueba y a las futuras URLs que procese nuestra aplicación web. Esto evita la "fuga de datos" (data leakage).

In [ ]:
# 4.1 Construcción del Pipeline y Entrenamiento

# Construimos el pipeline uniendo el preprocesador y el modelo de Regresión Logística
# solver='lbfgs': Es el algoritmo de optimización interno, muy eficiente para datasets como este.
# max_iter=1000: Aumentamos las iteraciones para asegurar que el modelo converja (encuentre el mínimo de la función de costo log-loss).
# class_weight='balanced': ¡Crucial! Le dice al modelo que preste más atención a la clase minoritaria (Phishing), 
# penalizando más los errores cometidos al predecir phishing.
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('modelo', LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42))
])

# Entrenamos el modelo usando UNICAMENTE los datos de entrenamiento (X_train, y_train)
pipeline_lr.fit(X_train, y_train)

print("¡Modelo de Regresión Logística entrenado con éxito!")

### 4.2 Evaluación del Modelo
Como vimos en teoría, el *Accuracy* (Exactitud) no es suficiente cuando las clases están desbalanceadas (14% phishing). Un modelo que siempre diga "Benigno" tendría 85% de Accuracy pero sería inútil. 

Por eso, evaluamos usando:
- **Matriz de Confusión:** Para ver los Falsos Positivos (URLs seguras bloqueadas) y Falsos Negativos (Phishing que logró pasar).
- **Recall (Sensibilidad):** De todas las páginas que realmente eran Phishing, ¿cuántas logró detectar el modelo?
- **Precision:** De todas las que el modelo clasificó como Phishing, ¿cuántas realmente lo eran?
- **Curva ROC / AUC:** Muestra el equilibrio entre verdaderos positivos y falsos positivos.

In [ ]:
# Realizamos las predicciones sobre el conjunto de prueba (nunca visto por el modelo)
# pipeline_lr.predict() hace que los datos pasen primero por el ColumnTransformer (preproc.transform) 
# y luego por el modelo. Así se garantiza la disciplina de machine learning.
y_pred = pipeline_lr.predict(X_test)
y_proba = pipeline_lr.predict_proba(X_test)[:, 1] # Probabilidad porcentual de ser Phishing

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 1. Reporte de Clasificación
print("--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_test, y_pred, target_names=['Benigna (0)', 'Phishing (1)']))

# 2. Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benigna Predicha', 'Phishing Predicho'],
            yticklabels=['Benigna Real', 'Phishing Real'])
plt.title('Matriz de Confusión')
plt.show()

In [ ]:
# 3. Curva ROC y AUC
# El AUC (Área Bajo la Curva) nos dice qué tan bien el modelo separa ambas clases. 
# Un AUC de 0.5 es azar puro, un AUC cercano a 1.0 es excelente.
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Azar (AUC = 0.5)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR o Recall)')
plt.title('Curva ROC - Regresión Logística')
plt.legend(loc="lower right")
plt.show()

## 5. Reentrenamiento y Exportación del Modelo
Una vez que estamos satisfechos con el rendimiento en la evaluación (`X_test`), la teoría dice que para pasar a la aplicación web (producción) debemos **reentrenar el pipeline con el 100% de los datos** (`X` e `y` completos).

Empaquetaremos este pipeline usando la librería `joblib`.

In [ ]:
# Reentrenamos el pipeline con TODOS los datos originales para no desperdiciar el 20% de test.
pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('modelo', LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42))
])

pipeline_final.fit(X, y)
print("Modelo final reentrenado con el 100% de los datos.")

# Exportamos el pipeline en formato .pkl
nombre_archivo = 'modelo_lr_phishing.pkl'
joblib.dump(pipeline_final, nombre_archivo)
print(f"\n¡El modelo ha sido exportado exitosamente como '{nombre_archivo}'!")
print("Este es el archivo exacto que cargará nuestra interfaz visual Streamlit para predecir URLs nuevas.")